In [16]:
!kaggle competitions download -c cifar-10


401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles


In [21]:
import os
import torch
from torchvision import datasets, transforms
root = r"C:\Users\User\Downloads\cifar-10"
print("root inside",os.listdir(root))

root inside ['sampleSubmission.csv', 'test.7z', 'train.7z', 'trainLabels.csv']


In [23]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset

class CIFAR10KaggleDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

        # 固定類別順序
        self.classes = [
            "airplane", "automobile", "bird", "cat", "deer",
            "dog", "frog", "horse", "ship", "truck"
        ]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_id = self.data.iloc[idx, 0]      # id
        label_name = self.data.iloc[idx, 1]  # label

        img_path = os.path.join(self.img_dir, f"{img_id}.png")
        image = Image.open(img_path).convert("RGB")

        label = self.classes.index(label_name)

        if self.transform:
            image = self.transform(image)

        return image, label

In [35]:
#testdata
class CIFAR10KaggleTestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform

        self.image_ids = []
        for name in os.listdir(img_dir):
            if name.endswith(".png"):
                self.image_ids.append(int(name[:-4]))

        self.image_ids.sort()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = os.path.join(self.img_dir, f"{img_id}.png")

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, img_id

In [28]:
from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.ToTensor()
])

dataset = CIFAR10KaggleDataset(
    csv_file=r"C:\Users\User\Downloads\cifar-10\trainLabels.csv",
    img_dir=r"C:\Users\User\Downloads\cifar-10\train\train",
    transform=transform
)

trainDataLoader = DataLoader(dataset, batch_size=32, shuffle=True)

images, labels = next(iter(trainDataLoader))
print(images.shape)   # [32, 3, 32, 32]
print(labels.shape)   # [32]
print(labels[:5])

torch.Size([32, 3, 32, 32])
torch.Size([32])
tensor([7, 5, 9, 4, 6])


In [ ]:
from torchvision.models.vision_transformer import VisionTransformer
import torch
import torch
import torch.nn as nn
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VisionTransformer(
    image_size=32, #圖片大小
    patch_size=4,  #每個 patch
    num_layers=6,  #encoder 層數
    num_heads=8,   #attention heads
    hidden_dim=256,#token embedding 維度
    mlp_dim=512,    ## MLP hidden dim
    num_classes=10
)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(),lr=1e-3)


In [33]:
from tqdm.auto import tqdm
num_epochs = 10
model = model.to(device)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    pbar = tqdm(trainDataLoader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=True)

    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{correct/total:.4f}"
        })

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss:.4f}, Acc: {correct/total:.4f}")


Epoch 1/10:   0%|          | 0/1563 [00:00<?, ?it/s]

Epoch 1/10, Loss: 2539.7062, Acc: 0.4065


Epoch 2/10:   0%|          | 0/1563 [00:00<?, ?it/s]

Epoch 2/10, Loss: 2479.0183, Acc: 0.4225


Epoch 3/10:   0%|          | 0/1563 [00:00<?, ?it/s]

...

Epoch 8/10, Loss: 2326.6300, Acc: 0.4629


Epoch 9/10:   0%|          | 0/1563 [00:00<?, ?it/s]

Epoch 9/10, Loss: 2305.3148, Acc: 0.4675


Epoch 10/10:   0%|          | 0/1563 [00:00<?, ?it/s]

Epoch 10/10, Loss: 2241.5923, Acc: 0.4816


In [40]:
classes = [
            "airplane", "automobile", "bird", "cat", "deer",
            "dog", "frog", "horse", "ship", "truck"
        ]
testdataset = CIFAR10KaggleTestDataset(
    img_dir=r"C:\Users\User\Downloads\cifar-10\test\test",
    transform=transform
)
testloader  = DataLoader(testdataset, batch_size=32, shuffle=False)
model.eval()
preds_test  = []
ids=[]
with torch.no_grad():
    for x, id in tqdm(testloader):
        x = x.to(device)
        y = model(x)
        pred = y.argmax(dim=1).cpu().tolist()
        preds_test .extend(pred)
        ids.extend(id.tolist())

  0%|          | 0/9375 [00:00<?, ?it/s]

In [41]:
import pandas as pd


pred_labels = [classes[p] for p in preds_test]

submission = pd.DataFrame({
    "id": ids,
    "label": pred_labels
})

submission.to_csv("submission_vit.csv", index=False)

In [42]:
print(len(testdataset))

300000
